# Group, summarize, and compare · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 16. Group, summarize, and compare

Distinguish group summaries, row-aligned transformations, and group filters.

Before you start: You can select columns, use summary methods, and create new columns.

Official reference: https://pandas.pydata.org/docs/user_guide/groupby.html

### M16C01. One question for each group

`groupby()` splits rows according to a label, then a summary method calculates one result per group. Named aggregation lets you choose readable output names. Here each campus receives its own row count and mean score.

**Read:** What does each row of the result represent?

Your explanation: _write here_.

**Predict:** How many rows should result contain?

Your explanation: _write here_.

**Explain:** Why use as_index=False?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many rows should result contain?
df = students()
result = df.groupby("Campus", as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))

**Your turn:** Group by Program instead. Store a DataFrame named result with Program as a normal column and named summary columns Students, Median_score, and Highest_score.

For the check, keep these result names: `result`.

**Hint:** Use named aggregations with size, median, and max.

**Watch for this:** A mean without its supporting count can hide a very small group.

In [ ]:
# Change this code to complete the task.
# Question: Group by Program instead. Store a DataFrame named result with Program as a normal column and named summary columns Students, Median_score, and Highest_score.
df = students()
result = df.groupby("Campus", as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))

**Quick check:** How many values does an ordinary mean aggregation return for one group?

1. One summary value
2. One value for every original row
3. One value for every cell

**Explain what you learned:** Summarize each group with readable labels and counts.

### M16C02. Multiple keys and missing group labels

Grouping by a list of columns forms one group for each observed combination. By default, rows with a missing group key are excluded. `dropna=False` keeps those rows in a separate missing-label group so the totals can be audited.

**Read:** What does one result row represent now?

Your explanation: _write here_.

**Predict:** How many student records are represented in total?

Your explanation: _write here_.

**Explain:** What would the default dropna=True do to S001?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many student records are represented in total?
df = students()
df.loc[0, "Campus"] = None
result = df.groupby(["Campus", "Program"], dropna=False, as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))
print("Total represented:", result["Students"].sum())

**Your turn:** Run the same grouping with dropna=True. Compare its represented row total with len(df), and store the grouped table in result.

For the check, keep these result names: `df`, `result`.

**Hint:** Keep the original table; add the group sizes and compare the sum.

**Watch for this:** When group keys are categorical, set observed explicitly if you need predictable inclusion of unused categories.

In [ ]:
# Change this code to complete the task.
# Question: Run the same grouping with dropna=True. Compare its represented row total with len(df), and store the grouped table in result.
df = students()
df.loc[0, "Campus"] = None
result = df.groupby(["Campus", "Program"], dropna=False, as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))
print("Total represented:", result["Students"].sum())

**Quick check:** Which option retains rows whose grouping key is missing?

1. dropna=False
2. dropna=True
3. ascending=False

**Explain what you learned:** Reconcile grouped totals with the original table.

### M16C03. A summary for every original row

`transform()` returns values aligned with the original rows. A group mean can therefore be attached to every student in that group. This makes comparisons with a group benchmark possible without reducing the table to one row per group.

**Read:** How many rows does result keep?

Your explanation: _write here_.

**Predict:** Will all AI students have the same Program_mean?

Your explanation: _write here_.

**Explain:** Why would assigning a two-row group mean Series directly be risky?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will all AI students have the same Program_mean?
df = students()
df["Program_mean"] = df.groupby("Program")["Score"].transform("mean")
df["Difference"] = df["Score"] - df["Program_mean"]
result = df[["Student_ID", "Program", "Score", "Program_mean", "Difference"]]
print(result.round(2))

**Your turn:** Calculate the median score within each campus in a column named Campus_median. Keep only students scoring strictly above that median. Store Student_ID, Campus, Score, and Campus_median in result.

For the check, keep these result names: `result`.

**Hint:** Use groupby("Campus")["Score"].transform("median") to build the benchmark.

**Watch for this:** A group-relative label and a fixed-threshold label answer different questions.

In [ ]:
# Change this code to complete the task.
# Question: Calculate the median score within each campus in a column named Campus_median. Keep only students scoring strictly above that median. Store Student_ID, Campus, Score, and Campus_median in result.
df = students()
df["Program_mean"] = df.groupby("Program")["Score"].transform("mean")
df["Difference"] = df["Score"] - df["Program_mean"]
result = df[["Student_ID", "Program", "Score", "Program_mean", "Difference"]]
print(result.round(2))

**Quick check:** Which operation returns a group statistic aligned to every original row?

1. agg
2. transform
3. drop_duplicates

**Explain what you learned:** Use transform when the result belongs beside the original records.

### M16C04. Keep whole groups and control unused categories

A group filter keeps or removes whole groups based on a group-level rule. Categorical columns can also contain labels with no rows. `observed=True` limits a group result to categories actually present in the data.

**Read:** Does declaring CS as a category create CS student records?

Your explanation: _write here_.

**Predict:** Will CS appear in this grouped result?

Your explanation: _write here_.

**Explain:** How is filtering whole groups different from filtering scores >= 80?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will CS appear in this grouped result?
df = students()
df["Program"] = pd.Categorical(df["Program"], categories=["AI", "IT", "CS"])
result = df.groupby("Program", observed=True)["Score"].agg(["size", "mean"])
print(result.round(2))

**Your turn:** Start from students(). Keep all students in programs whose mean Score is at least 80 using groupby().filter().

For the check, keep these result names: `result`.

**Hint:** The filter function receives each group DataFrame; compare its Score mean with 80.

**Watch for this:** Unused categories and missing labels are different: observed controls unused categories; dropna controls missing keys.

In [ ]:
# Change this code to complete the task.
# Question: Start from students(). Keep all students in programs whose mean Score is at least 80 using groupby().filter().
df = students()
df["Program"] = pd.Categorical(df["Program"], categories=["AI", "IT", "CS"])
result = df.groupby("Program", observed=True)["Score"].agg(["size", "mean"])
print(result.round(2))

**Quick check:** A program passes the group-mean rule. Which records are kept by groupby().filter()?

1. Only records above the threshold
2. All records in that program
3. Only the highest-scoring record

**Explain what you learned:** Separate group inclusion rules from row inclusion rules.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates